# Silver layer

## Customer information

Information about each customer:
* Id and key
* Name
* Surname 
* Maritial status 
* Gender
* Row creation date

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, upper, now, isnull, current_date, max, min, isnotnull, length
from pyspark.sql.types import *

# Read and Transform

## Correct Strings

For each string type column, apply trim to remove unnecessary spaces

In [0]:
df = spark.read.table("db_project.bronze.crm_cust_info")
# df.display()
fields = df.schema.fields
# display(fields)
# display(fields[0])
for field in fields:
    if isinstance(field.dataType, StringType):
        display(field.dataType)
        df = df.withColumn(field.name, trim(col(field.name)))
df.limit(100).display()

## Check maritial status

In [0]:
df.select("cst_marital_status").distinct().display()

Modify each status and provide full names

In [0]:
df = df.withColumn('cst_marital_status', 
                   F.when(col('cst_marital_status').isin(['Married', 'M']), 'Married')
                   .when(col('cst_marital_status').isin(['Single', 'S']), 'Single')
                   .otherwise('n/a')
                   )
df.limit(1000).display()

Look up why maritial status may be unknown

In [0]:
# df.where(col('cst_marital_status') == 'n/a').display()
test_status = df.where(df.cst_marital_status == 'n/a')
test_status.display()

## Check gender

Check distinct genders within the table

In [0]:
df.select("cst_gndr").distinct().display()

Modify gender names to their full forms

In [0]:
df = df.withColumn("cst_gndr", 
                   F.when(upper(df.cst_gndr).isin(["F", "FEMALE"]), "Female")
                   .when(upper(df.cst_gndr).isin(["M", "MALE"]), "Male")
                   .otherwise('n/a'))

## Check dates

Check customer creation dates with min and max functions

In [0]:
test_date_v1 = df.select(
    max("cst_create_date").alias("newest_creation_date"),
    min("cst_create_date").alias("oldest_creation_date")
)
test_date_v1.display()

Dates within the norm

## Check nulls

In [0]:
test_nulls = df.where(
    df.cst_id.isNull() |
    df.cst_key.isNull() |
    df.cst_firstname.isNull() |
    df.cst_lastname.isNull() |
    df.cst_create_date.isNull()
)
test_nulls.display()

Possible incorrect recods within the table:
* Rows with customer id as unknown have incorrect keys and don't have creation dates, possibly incorrect records - to be removed
* Other rows with Null values seem correct but simply lacking information - to be left

In [0]:
df = df.where(df.cst_id.isNotNull())

## Check customer key and id

See if either key or id have records with different lengths

In [0]:
test_id_key = df.select(
    F.round(F.avg(length("cst_key")), 5).alias("average_key_length"),
    F.round(F.avg(length("cst_id")), 5).alias("average_id_length")
)
test_id_key.display()

In [0]:
df.limit(100).display()

# Write table silver.crm_cust_info

Everything looks ok, so save DataFrame into Delta Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", True).format("delta").saveAsTable("db_project.silver.crm_cust_info")